<a href="https://colab.research.google.com/github/bregiang/IS4460-Project-Template/blob/main/Giang_Breanna_Assignment2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> ### Note on Labs and Assignments:
>
> 🔧 Look for the **wrench emoji** — it marks code you must change. Routine run-only cells do not use it.
>
> 🖊 Look for the **writing emoji** — it marks analysis you must write.
>
> These sections are graded and are not optional.
>

# Module 2 Assignment 2: Local Model Task Portfolio

**Notebook:** Student Template  
**Student:** Edit the configuration cell  
**Required models:** `gemma3:1b`, `gemma3:4b`, `llama3.2:1b`, and `llama3.2:3b`

This notebook supports the complete Assignment 2 workflow: four direct business
tasks, prompt revision, a controlled four-model comparison, evidence-based scoring, reflection, and AI-use disclosure.


## Student Introduction: What is this Assignment Is About?

This assignment asks you to work directly with small language models running on your own machine via Ollama. You will practice two core skills:

1. **Prompt engineering** — writing and iteratively improving instructions that guide a model toward a useful business output.
2. **Model evaluation** — systematically comparing four models on the same task and scoring them with evidence.

**What you will produce:**

- **Part 1:** Four business tasks (extraction, summarization, drafting, classification). For each task, you write an initial zero-shot instruction, run it, diagnose a specific weakness in the output, revise the instruction using a named prompting strategy, run it again, and evaluate the improvement.
- **Part 2:** A controlled four-model comparison using a shared service-request packet. You design a single instruction, run it on all four models without changing anything, then score each model across four dimensions with evidence from their outputs.
- **Part 3:** A 350–500 word reflection answering seven specific questions about what you observed.

**Before you start:**
1. Run the initial code blocks to install Ollama and start it.
2. Replace `"Breanna Giang"` in the configuration cell below with your actual name.
3. Run all cells from top to bottom in order.

The notebook will raise an error and stop if any required `TODO` is still present when you try to run a model — this is intentional so you do not accidentally submit incomplete work.

## Important Instructions

1. Read the assignment before editing this notebook.
2. Edit only cells marked for student work.
3. Do not change the comparison source packet, model list, or shared settings.
4. Preserve the first output from every run.
5. Before submitting, restart the kernel and run all cells from top to bottom.

The template intentionally raises a clear error when a required `TODO` remains.


## Setup Ollama

This notebook will download, install and start [Ollama](https://ollama.com/download). The four default model downloads require approximately 8 GB in total.





### What is Ollama?

Ollama is a tool that lets you run AI language models on your own computer instead of only using an online service like ChatGPT.

For a beginner, you can think of it as a local “AI model manager.” It helps you download a model, start it, and send it prompts. For example, instead of calling an online API from OpenAI, Google, or Anthropic, you can call an Ollama model running on your laptop or server.

The basic idea is:



*   You install Ollama.
*   You download a model, such as Llama, Gemma, or Mistral.
*   You send text to the model.
*  The model sends text back.


Why are we doing this rather than using Claude or ChatGPT?

* Reproducability. The notebook allows students to all follow the same steps and instructions.
* Cost: API access to Anthropic,OpenAI, Google models is not free. These models are free to run. The trade-off? (There always is one).
* What do we sacrifice for using the free models? Quality of responses and speed. We'll be using CPUs since these models are small language models (SLMs) rather than large language models (LLMs)






In [1]:
import subprocess
import time

In [2]:
# Download and install Ollama (Google Colab only — skip if running locally)
install_zstd = subprocess.run(
    "sudo apt-get install zstd",
    shell=True,
    capture_output=True,
    text=True,
)

install_zstd

CompletedProcess(args='sudo apt-get install zstd', returncode=0, stdout='Reading package lists...\nBuilding dependency tree...\nReading state information...\nThe following NEW packages will be installed:\n  zstd\n0 upgraded, 1 newly installed, 0 to remove and 57 not upgraded.\nNeed to get 603 kB of archives.\nAfter this operation, 1,695 kB of additional disk space will be used.\nGet:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]\nFetched 603 kB in 2s (363 kB/s)\nSelecting previously unselected package zstd.\n(Reading database ... \n(Reading database ... 5%\n(Reading database ... 10%\n(Reading database ... 15%\n(Reading database ... 20%\n(Reading database ... 25%\n(Reading database ... 30%\n(Reading database ... 35%\n(Reading database ... 40%\n(Reading database ... 45%\n(Reading database ... 50%\n(Reading database ... 55%\n(Reading database ... 60%\n(Reading database ... 65%\n(Reading database ... 70%\n(Reading database ... 75%\n(Reading datab

In [3]:
# RUN THIS CELL. YOU SHOULD SEE Ollama installed and Ollama server is running messages.

# Download and install Ollama
install = subprocess.run(
    "curl -fsSL https://ollama.com/install.sh | sh",
    shell=True,
    capture_output=True,
    text=True,
)
if install.returncode != 0:
    raise RuntimeError(f"Ollama installation failed:\n{install.stderr}")
print("Ollama installed.")

# Start the Ollama server as a background process
subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

# Give the server a few seconds to initialize before any requests are made
time.sleep(3)
print("Ollama server is running.")

Ollama installed.
Ollama server is running.


In [4]:
# RUN THIS CELL

from datetime import datetime
from hashlib import sha256
from time import perf_counter
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen
import json
import textwrap

from IPython.display import Markdown, display
import subprocess
import time


REFERENCE_MODE = False
AUTO_PULL_MODELS = True
OLLAMA_BASE_URL = "http://localhost:11434"

REQUIRED_MODELS = [
    "gemma3:1b",
    "gemma3:4b",
    "llama3.2:1b",
    "llama3.2:3b",
]
BASELINE_MODEL = "gemma3:1b"
GENERATION_OPTIONS = {
    "temperature": 1,
    #"seed": 4490,
    "num_ctx": 8192,
    "num_predict": 900,
}




In [5]:
# RUN THIS CELL

REFERENCE_OUTPUTS = {}


def require_finished(label, value):
    """Stop before a model run when a required student field is unfinished."""
    if value is None or "TODO" in str(value):
        raise ValueError(f"Complete {label} before running this cell.")


def ollama_request(path, payload=None, timeout=120):
    """Send a JSON request to the local Ollama service."""
    data = None if payload is None else json.dumps(payload).encode("utf-8")
    request = Request(
        f"{OLLAMA_BASE_URL}{path}",
        data=data,
        headers={"Content-Type": "application/json"},
        method="GET" if payload is None else "POST",
    )
    try:
        with urlopen(request, timeout=timeout) as response:
            return json.loads(response.read().decode("utf-8"))
    except HTTPError as exc:
        details = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(
            f"Ollama returned HTTP {exc.code}: {details}"
        ) from exc
    except URLError as exc:
        raise RuntimeError(
            "Cannot connect to Ollama at http://localhost:11434. "
            "Install and start Ollama, then rerun this cell."
        ) from exc


def chat_once(model, prompt, run_key):
    """Run one independent prompt and return content plus observable metadata."""
    if REFERENCE_MODE:
        fixture = REFERENCE_OUTPUTS[run_key]
        return {
            "model": model,
            "run_key": run_key,
            "recorded_at": fixture["recorded_at"],
            "elapsed_seconds": fixture["elapsed_seconds"],
            "content": fixture["content"],
            "prompt_eval_count": None,
            "eval_count": None,
            "reference_fixture": True,
        }

    started_at = datetime.now().astimezone().isoformat(timespec="seconds")
    start = perf_counter()
    response = ollama_request(
        "/api/chat",
        {
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
            "stream": False,
            "keep_alive": 0,
            "options": GENERATION_OPTIONS,
        },
        timeout=900,
    )

    #print(f"Generation Options:{GENERATION_OPTIONS}")

    elapsed = round(perf_counter() - start, 2)
    return {
        "model": model,
        "run_key": run_key,
        "recorded_at": started_at,
        "elapsed_seconds": elapsed,
        "content": response["message"]["content"].strip(),
        "prompt_eval_count": response.get("prompt_eval_count"),
        "eval_count": response.get("eval_count"),
        "reference_fixture": False,
    }


def display_record(record):
    metadata = (
        f"**Model:** `{record['model']}`  \n"
        f"**Recorded:** {record['recorded_at']}  \n"
        f"**Elapsed:** {record['elapsed_seconds']} seconds"
    )
    display(Markdown(metadata))
    display(Markdown(record["content"]))


def compose_prompt(instruction, source):
    return (
        instruction.strip()
        + "\n\nSOURCE\n------\n"
        + source.strip()
    )


def run_part1(task_key, stage, instruction, source):
    require_finished(f"{task_key} {stage} instruction", instruction)
    return chat_once(
        BASELINE_MODEL,
        compose_prompt(instruction, source),
        f"part1_{task_key}_{stage}",
    )


In [6]:
# RUN THIS CELL
# You should see
# Attempting to pull model: gemma3:1b
# Successfully pulled gemma3:1b.
# etc.

# Check and pull required models if AUTO_PULL_MODELS is True
if AUTO_PULL_MODELS:
    print(f"Checking and pulling required models: {REQUIRED_MODELS}")
    for model_name in REQUIRED_MODELS:
        print(f"Attempting to pull model: {model_name}")
        # Use subprocess to run ollama pull command
        pull_result = subprocess.run(
            f"ollama pull {model_name}",
            shell=True,
            capture_output=True,
            text=True,
        )
        if pull_result.returncode != 0:
            print(f"Failed to pull {model_name}:\n{pull_result.stderr}")
        else:
            print(f"Successfully pulled {model_name}.")
        time.sleep(1) # Give a moment between pulls

Checking and pulling required models: ['gemma3:1b', 'gemma3:4b', 'llama3.2:1b', 'llama3.2:3b']
Attempting to pull model: gemma3:1b
Successfully pulled gemma3:1b.
Attempting to pull model: gemma3:4b
Successfully pulled gemma3:4b.
Attempting to pull model: llama3.2:1b
Successfully pulled llama3.2:1b.
Attempting to pull model: llama3.2:3b
Successfully pulled llama3.2:3b.


# Part 1: Direct AI Task Portfolio

In this part you complete four independent business tasks using the baseline model (`gemma3:1b`). Each task gives you a realistic business scenario and a source text. Your job is to engineer the prompt that produces the most useful output.

**How each task section works:**

1. **Write a zero-shot instruction** in the first code cell. Zero-shot means plain directions only — no examples, no step-by-step reasoning prompts. The code comment already labels it for you.
2. **Run the cell and preserve the output.** Do not delete or re-run the initial output before recording it. Your submission must show the original output.
3. **Diagnose the output** in the markdown cell that follows. Compare what the model produced against the source text. Identify one specific, evidence-based weakness (something missing, wrong, or poorly formatted). Quote or paraphrase both the source and the output.
4. **Write a revised instruction** in the second code cell, addressing the weakness you identified. For **at least two of the four tasks**, apply a named prompting strategy and identify it at the top of your instruction string.
5. **Evaluate the revision** in the final markdown cell. Explain whether the revision materially improved the output, what role the chosen strategy played, and what decisions still require a human's judgment.

**Named strategies you may apply for revised instructions:**

| Strategy | What it means |
|---|---|
| **Few-shot** | Include one or more examples of the expected input/output pattern before the task. |
| **Chain-of-thought** | Instruct the model to reason step by step before giving its final answer. |
| **Persona** | Assign the model a specific role or professional background before the task. |
| **Zero-shot** | Plain directions only — acceptable when the initial output already meets your standards. |

Preserve every initial output before revising an instruction.

In [7]:
PART1_SOURCES = {
  "extraction": "From: Maya Chen\nTo: Facilities Service Desk\nSubject: Loose handrail before Friday tour\n\nThe handrail in the east stairwell on floor 3 of Pioneer Hall is loose at the\nlower wall bracket. I noticed it at 9:15 a.m. on September 2. No one has been\ninjured, and the stairwell is still open. Please repair it before the visitor\ntour this Friday if possible. I can meet a technician after 1:00 p.m. Call me\nat extension 5521.",
  "summarization": "Customer Elena Ruiz reported that order OR-8841 was charged twice. The\noriginal $186.40 charge posted on August 6, and a second $186.40 charge posted\non August 8 after she refreshed the checkout page. The order itself arrived on\nAugust 10 and was correct. Agent Malik opened case CS-2197 on August 11 and\nasked Billing to investigate. Billing has not yet confirmed whether the second\nentry is a settled charge or a temporary authorization. Elena wants the second\ncharge removed if it settled, but she does not want the order canceled. She\nasked for an update by August 13 because her card payment is due August 14.",
  "drafting": "Supplier Northstar Filtration notified Procurement that shipment NF-771,\ncontaining 12 replacement filters, will arrive August 19 instead of August 14.\nThe plant currently has approximately four days of filter inventory at normal\nusage. Northstar offered expedited shipping for an additional fee, but\nProcurement has not approved that option. Operations is checking whether usage\ncan be reduced safely. The plant manager needs a status update today. No\nproduction shutdown has been scheduled.",
  "classification": "Routing categories:\n- IT Support: computers, software, networks, and accounts\n- Security Access: badges, controlled doors, and physical-access permissions\n- Facilities: building fixtures, utilities, and room conditions\n\nUrgency rules:\n- Urgent: an active safety issue or current business operation is blocked with\n  no workaround\n- Standard: future need, routine repair, or a workable temporary alternative\n\nRequest: \"My new analyst starts Monday. Her employee account works, but her\nbadge does not open the Finance Annex. I can meet her in the lobby and escort\nher on the first day if needed. Please add normal weekday access before 8:00\na.m. Monday. The request does not include the analyst's employee ID or the\nmanager's access approval record.\" "
}


## Part 1.1: Extraction

In this section you are fulfilling the role of an AI Automation Specialist working as a consultant for a local facilities management operation. They have been struggling with the amount of time it takes to read and synthesize information from unstructured emails from customers at various facilities. Your job is to extract key items from emails.

Your first prompt must be a **ZERO-SHOT PROMPT**. In the next section you will use other strategies to improve the output.

**Business user:** Facilities coordinator  
**Purpose:** Turn an emailed repair request into a consistent intake record.

**Provided input**

```text
From: Maya Chen
To: Facilities Service Desk
Subject: Loose handrail before Friday tour

The handrail in the east stairwell on floor 3 of Pioneer Hall is loose at the
lower wall bracket. I noticed it at 9:15 a.m. on September 2. No one has been
injured, and the stairwell is still open. Please repair it before the visitor
tour this Friday if possible. I can meet a technician after 1:00 p.m. Call me
at extension 5521.
```


#### TODO - INSTRUCT 🔧

In [8]:
# 🔧 TODO - INSTRUCT
# Strategy: zero-shot — clear directions only, no examples.
# Your GOAL here is to extract the following pieces of information from the email.
# Extract reporter name, location, problem description, date and time observed, urgency or deadline, and contact information."

initial_instruction_extraction = """
Extract the key information from the provided facilities repair request email and organize it into a consistent intake record.

Identify and return the following fields:
- Reporter Name
- Location
- Problem Description
- Date and Time Observed
- Urgency or Deadline
- Contact Information

Use only information explicitly stated in the email. Do not infer or invent missing information. Keep the extracted information concise and preserve important details such as the specific location, requested deadline, and technician availability.

"""

In [9]:
# Run this to generate output after updating your instruction. It will take at least 20 seconds to run each loop.

for i in range(3):

  initial_record_extraction = run_part1(
      task_key="extraction",
      stage="initial",
      instruction=initial_instruction_extraction,
      source=PART1_SOURCES["extraction"],
  )
  display_record(initial_record_extraction)

**Model:** `gemma3:1b`  
**Recorded:** 2026-09-01T22:35:28+00:00  
**Elapsed:** 138.32 seconds

Here's the structured information extracted from the email:

*   **Reporter Name:** Maya Chen
*   **Location:** Pioneer Hall, east stairwell, floor 3
*   **Problem Description:** Loose handrail in the lower wall bracket
*   **Date and Time Observed:** September 2 at 9:15 a.m.
*   **Urtgency/Deadline:** Immediate repair needed – before the visitor tour this Friday.
*   **Contact Information:** Extension 5521

**Model:** `gemma3:1b`  
**Recorded:** 2026-09-01T22:37:46+00:00  
**Elapsed:** 5.49 seconds

Okay, please provide the email text so I can extract the information for the intake record. Once you copy and paste the text, I will do exactly as requested.

**Model:** `gemma3:1b`  
**Recorded:** 2026-09-01T22:37:52+00:00  
**Elapsed:** 4.84 seconds

Okay, here’s a structured intake record extracting the key information from the email:

*   **Reporter Name:** Maya Chen
*   **Location:** East stairwell, floor 3, Pioneer Hall
*   **Problem Description:** Loose handrail at the lower wall bracket.
*   **Date & Time Observed:** 9:15 a.m. on September 2nd
*   **Urgency/Deadline:** High – Needs repair prior to visitor tour this Friday
*   **Contact Information:**  Extension 5521 - Call Maya Chen

### TODO - REFLECT 🖊

Initial-Output Diagnosis and Revision Plan

Specific weakness in the initial output:

The initial prompt successfully identifies the six required fields, but it does not specify a strict output format. This could cause the model to return the information in inconsistent formats across different repair requests. For example, the source email contains several distinct details like “east stairwell on floor 3 of Pioneer Hall,” “loose at the lower wall bracket,” and “9:15 a.m. on September 2” that need to be consistently separated into the appropriate intake fields. The initial instruction says to “organize it into a consistent intake record,” but it does not define exactly how that record should be structured.

Speed of Output:

CPU: 138.32 seconds on average
GPU: 4.84 seconds on average

Enter the average times recorded when you ran the initial prompt on the CPU and GPU.

Planned instruction change:

The revised instruction will require the model to return the extracted information using a fixed labeled format. It will also explicitly instruct the model to distinguish between the location, problem description, observation date/time, deadline or urgency, and contact information. This should make the results more consistent and easier for a facilities coordinator to use as an intake record.

Prompting strategy for the revision:

Few-shot prompting will be used for the revision. This strategy is appropriate because the main weakness is consistency in how the information is organized. Providing one or more examples of correctly formatted repair-request extractions will show the model exactly what the desired intake record should look like, while still allowing it to apply the pattern to new emails.

### Revise your prompt with a strategy to address the issue you noted above with the model output quality.

### TODO - INSTRUCT 🔧

In [10]:
# 🔧 TODO. Write the revised instruction.
revised_instruction_extraction = """

Extract the key information from the provided facilities repair request email and organize it into a consistent intake record.

Return the information using exactly these six fields:
- Reporter Name:
- Location:
- Problem Description:
- Date and Time Observed:
- Urgency or Deadline:
- Contact Information:

Use only information explicitly stated in the email. Do not infer, assume, or invent information. Include all relevant details from the email under the most appropriate field. Keep each field concise but specific.

Example:

Email: "John Smith reported that a leaking pipe was found under the sink in Room 204 on March 3 at 10:30 a.m. No injuries occurred. Please fix it before March 5. John can be reached at extension 1234."

Output:
Reporter Name: John Smith
Location: Room 204, under the sink
Problem Description: Leaking pipe
Date and Time Observed: March 3 at 10:30 a.m.
Urgency or Deadline: Repair requested before March 5
Contact Information: Extension 1234

Now extract the same six fields from the provided email.
"""


In [11]:
# run this to create the output. It will take at least 20 seconds to run each loop.
for i in range(3):

  revised_record_extraction = run_part1(
      task_key="extraction",
      stage="revised",
      instruction=revised_instruction_extraction,
      source=PART1_SOURCES["extraction"],
  )
  display_record(revised_record_extraction)

**Model:** `gemma3:1b`  
**Recorded:** 2026-09-01T22:47:26+00:00  
**Elapsed:** 4.52 seconds

Here's the intake record derived from the provided text:

- Reporter Name: Maya Chen
- Location: east stairwell on floor 3 of Pioneer Hall
- Problem Description: Loose handrail
- Date and Time Observed: September 2, 9:15 a.m.
- Urgency or Deadline: Repairs requested before the visitor tour this Friday
- Contact Information: Extension 5521

**Model:** `gemma3:1b`  
**Recorded:** 2026-09-01T22:47:30+00:00  
**Elapsed:** 4.4 seconds

Reported: Maya Chen
Location: East stairwell on floor 3, PioneerHall
Problem Description: Loose handrail
Date and Time Observed: September 2 at 9:15 a.m.
Urgency or Deadline: Requires immediate repair before visitor tour this Friday
Contact Information: Extension 5521

**Model:** `gemma3:1b`  
**Recorded:** 2026-09-01T22:47:35+00:00  
**Elapsed:** 5.44 seconds

Okay, let’s process the provided email and extract the information into the desired format.

Reporter Name: Maya Chen
Location: East stairwell on floor 3 of Pioneer Hall
Problem Description: Loose handrail
Date and Time Observed: September 2 at 9:15 a.m.
Urgency or Deadline: Please repair it before the visitor tour this Friday if possible
Contact Information: Extension 5521

Improvement and Human Review

Effect of the revision:

The revision materially improved the output's consistency and completeness. All three runs extracted the six required pieces of information, including Maya Chen, the Pioneer Hall location, the loose handrail, the September 2 observation time, the Friday visitor-tour deadline, and extension 5521. The fixed six-field format also made the information easier to scan and use as a facilities intake record.

Role of the prompting strategy:

The few-shot strategy contributed by giving the model an example of the desired structure and level of detail. This helped guide the model toward consistently identifying and labeling the six required fields instead of simply summarizing the email. However, the outputs still showed some variation, such as “Requires immediate repair” in one output, which is stronger than the source's “if possible.”

Human review still required:

A person should verify important facts before taking action, especially the urgency and deadline, exact location, and whether the repair requires immediate safety precautions. The model should not independently decide that a repair is “immediate” when the email only requests completion before Friday “if possible.” A facilities coordinator should also determine the appropriate technician, work order priority, and any necessary safety measures.

Output Variability:

The three outputs were fairly consistent, with all three identifying the required information correctly. However, there were small differences in formatting and wording, particularly around urgency. All 3 outputs could be used effectively as intake records, although the second and third would benefit from minor human editing because they were less cleanly formatted and the second slightly overstated the urgency.

## Part 1.2: Summarization

**Business user:** Customer-service supervisor  
**Purpose:** Prepare a concise escalation summary without losing financial details.

**Provided input**

```text
Customer Elena Ruiz reported that order OR-8841 was charged twice. The
original $186.40 charge posted on August 6, and a second $186.40 charge posted
on August 8 after she refreshed the checkout page. The order itself arrived on
August 10 and was correct. Agent Malik opened case CS-2197 on August 11 and
asked Billing to investigate. Billing has not yet confirmed whether the second
entry is a settled charge or a temporary authorization. Elena wants the second
charge removed if it settled, but she does not want the order canceled. She
asked for an update by August 13 because her card payment is due August 14.
```


**Your task:** A customer-service supervisor needs a concise escalation summary to hand off to a billing team. The model should condense the case above without losing any financially important detail — amounts, dates, case numbers, and the customer's stated deadline all matter.

**What to do in the cells below:**
- **First code cell:** Replace the `TODO` with your zero-shot instruction. Specify the audience (the billing team), the required level of detail, and any format constraints (length, structure). Run the cell and leave the output visible.
- **Diagnosis markdown cell:** Compare the output against the source. Identify one specific weakness — for example, a missing amount, a dropped date, or a format that buries the urgency.
- **Second code cell:** Write your revised instruction addressing that weakness. Remember: at least two of your four tasks must apply a named strategy with an explanation.
- **Evaluation markdown cell:** Explain what improved, what the strategy contributed, and which facts in the summary would need human verification before the billing team acts on them.

### TODO - INSTRUCT 🔧

In [12]:
# 🔧 Strategy: zero-shot — clear directions only, no examples.
initial_instruction_summarization = """
Summarize the customer case for the billing team as a concise escalation summary.

Include all financially important details from the source, especially:
- Customer name
- Order number
- Amount of each charge
- Dates of both charges
- What caused the second charge, if stated
- Order delivery date and whether the order was correct
- Customer-service case number
- Date the case was opened
- Current status of the billing investigation
- What the customer wants
- The customer's requested update deadline and relevant payment due date

Do not omit, change, or infer financial facts. Clearly distinguish confirmed facts from information that Billing has not yet verified.

Format the response as one short paragraph followed by a brief "Action Needed" line. Keep the total response under 150 words. Write for a billing team that needs to quickly understand the issue and determine the next step.
"""

In [13]:
for i in range(3):

  initial_record_summarization = run_part1(
      task_key="summarization",
      stage="initial",
      instruction=initial_instruction_summarization,
      source=PART1_SOURCES["summarization"],
  )
  display_record(initial_record_summarization)

**Model:** `gemma3:1b`  
**Recorded:** 2026-09-01T22:52:39+00:00  
**Elapsed:** 5.39 seconds

**Scenario Summary: Elena Ruiz – Order OR-8841 Billing Dispute**

Confirmation of a double charge for order OR-8841, totaling $186.40 on August 6th and a subsequent $186.40 on August 8th, following a customer’s customer service request.  The original charge on August 6th was confirmed as correct, but subsequent charges are currently unexplained and require verification. The customer requests that the second $186.40 charge be removed if it’s a settlement and denies cancellation of the original order.  Elena's requested update deadline is August 13th, and her card payment is due August 14th.  The billing team is actively investigating the discrepancy and needs to determine whether this is a genuine issue requiring resolution, a temporary authorization, or a fraudulent activity and has not yet confirmed the status of both charges to client Elena. 

**Action Needed:** Verify the validity of the second charge against the original order and the customer's account history to confirm the charge.

**Model:** `gemma3:1b`  
**Recorded:** 2026-09-01T22:52:44+00:00  
**Elapsed:** 6.27 seconds

**Escalation Summary**

Elena Ruiz, OR-8841, is currently facing two duplicate charges – $186.40 on August 6 and another $186.40 on August 8, impacting approximately $37.20 total. The order delivery date was August 10 and was confirmed to be correct. Customer Service Agent Malik opened case CS-2197 on August 11, requesting a billing investigation related to these two charges. Billing has not verified the nature of the second charge yet – it appears as a pending authorization. Ms. Ruiz is seeking removal of the second charge if it’s a settled transaction, but doesn’t require order cancellation. The customer's requested deadline for resolution is August 13 due to her August 14th payment due date, with a required investigation update.


**Action Needed:** Verify the second $186.40 charge against existing billing history and authorization logs to ascertain its validity.  Confirm the source of this authorization request and analyze potential authorization overrides or refunds.

**Model:** `gemma3:1b`  
**Recorded:** 2026-09-01T22:52:51+00:00  
**Elapsed:** 5.42 seconds

Please note the Customer Elena Ruiz initiated a billing investigation following an unexpected charge for two payments on order OR-8841 (amount $186.40, August 6, and $186.40, August 8). The order delivery date confirms the order was correct on August 10th. Client service case CS-2197 was opened on August 11th, with Billing tasked with investigating the two duplicate charges. Preliminary investigation has indicated a potential temporary authorization due to a refresh interaction, but billing is awaiting confirmation of this and a full settlement/removal of the duplicate charge. Elena is requesting a removal of the second charge and a full settlement of the balance due by August 13th, reflecting her due date of August 14th. 

Action Needed: Immediately validate the temporary authorization flagged for order OR-8841 to determine if it signifies a settled payment. Confirm the final settlement amount and pending payment due date with Elena to ensure reconciliation and provide a detailed resolution.

### TODO - REFLECT 🖊

Initial-Output Diagnosis and Revision Plan

**Specific weakness in the initial output:**

The outputs contained several **hallucinated or incorrect financial details**, which is especially risky for a billing handoff. For example, the second output says the two charges total “approximately $37.20,” even though tthe source clearly states two separate $186.40 charges. The outputs also incorrectly treated the second charge as a “pending authorization” or “potential temporary authorization,” even though the source explicitly says Billing has not yet confirmed whether it is a settled charge or temporary authorization. Some outputs also added unsupported actions such as investigating “fraudulent activity” or “authorization overrides.” These changes could cause Billing to act on information that was never confirmed.

**Planned instruction change:**

The revised instruction will explicitly require the model to **preserve every financial amount, date, case number, and customer deadline exactly as stated** and to clearly distinguish confirmed facts from unresolved information. It will also instruct the model not to calculate new amounts or introduce explanations, causes, or statuses that are not explicitly supported by the source.

**Prompting strategy for the revision:**

I will use few-shot prompting. Since the main problem is that the model is changing or inventing details, a carefully formatted example can demonstrate how to preserve exact financial information and label uncertain information without turning it into a confirmed fact. This should improve both accuracy and consistency while keeping the summary concise.


**Your task:** A procurement analyst needs to send the plant manager a status update about a delayed shipment. The draft must be factually accurate, avoid making commitments that have not been approved (e.g., expedited shipping has not been authorized), and convey appropriate urgency without overstating the risk.

**What to do in the cells below:**
- **First code cell:** Replace the `TODO` with your zero-shot instruction. Specify the intended recipient (the plant manager), the tone, and any constraints on what the draft should or should not commit to. Run the cell and leave the output visible.
- **Diagnosis markdown cell:** Review the draft for any facts that differ from the source, commitments the model made that are not supported, or tone and structure problems.
- **Second code cell:** Write your revised instruction. Consider whether a persona strategy (e.g., "You are a procurement analyst...") or chain-of-thought reasoning helps the model avoid unsupported commitments.
- **Evaluation markdown cell:** Explain what improved, what the strategy contributed, and what a human analyst must check before sending the draft.

### Revise your prompt with a strategy to address the issue you noted above with the model output quality.

### TODO - INSTRUCT 🔧

In [14]:
# 🔧 Identify your strategy at the start of the instruction string, e.g.: "Strategy: persona"
revised_instruction_summarization = """
Strategy: few-shot

Summarize the provided customer billing case for the billing team as a concise escalation summary.

Preserve all financially important information exactly as stated in the source, including:
- Customer name
- Order number
- Each individual charge amount
- Date of each charge
- Reason for the second charge, only if explicitly stated
- Order delivery date and status
- Customer-service case number and date opened
- Current Billing investigation status
- Customer's requested resolution
- Customer's requested update deadline
- Card payment due date

Do not calculate totals or introduce new financial figures. Do not describe an unconfirmed charge as settled, pending, fraudulent, or authorized. Clearly identify anything Billing has not yet confirmed.

Example:
Source: "A customer was charged $50 on March 1 and another $50 on March 3. Billing has not confirmed whether the second charge is settled or a temporary authorization. The customer wants the second charge removed if settled and requests an update by March 5."
Summary: "Customer was charged $50 on March 1 and an additional $50 on March 3. Billing has not confirmed whether the second $50 entry is a settled charge or temporary authorization. Customer wants the second charge removed if it settled and requests an update by March 5."

Use the same approach for the provided case. Write one concise paragraph followed by an "Action Needed" line. Keep the total response under 150 words. Base the summary only on information explicitly provided in the source.
"""

In [15]:
for i in range(3):

  revised_record_summarization = run_part1(
      task_key="summarization",
      stage="revised",
      instruction=revised_instruction_summarization,
      source=PART1_SOURCES["summarization"],
  )
  display_record(revised_record_summarization)

**Model:** `gemma3:1b`  
**Recorded:** 2026-09-01T22:56:08+00:00  
**Elapsed:** 5.06 seconds

Here's a concise escalation summary:

Elena Ruiz reported twice charges on order OR-8841. The original charge was August 6, and a second charge was August 8. The order arrived August 10 and is correct. Billing has not yet verified if the second charge is a settled or temporary authorization. Customer requested removal of the second charge upon settlement but no cancellation is needed.  She’s requesting an updated status by August 13th due to a card payment due on August 14th.

Action Needed: Review payment confirmations for OR-8841 to determine settlement status and urgency of response.

**Model:** `gemma3:1b`  
**Recorded:** 2026-09-01T22:56:13+00:00  
**Elapsed:** 5.28 seconds

“Elena Ruiz notified of duplicate charges on Order OR-8841. One charge of $186.40 occurred on August 6, and a second $186.40 on August 8 after a page refresh. The order delivered on August 10 and is valid. Billing is currently investigating the second charge, seeking confirmation of its status before the payment due date on August 14. Currently, Billing has not confirmed whether the second charge is settled or an authorization.”

**Model:** `gemma3:1b`  
**Recorded:** 2026-09-01T22:56:19+00:00  
**Elapsed:** 4.56 seconds

“Elena Ruiz reported two duplicate charges on OR-8841, appearing on August 6 and August 8. The order delivered on August 10, demonstrating the charge status is accurate. Billing is currently investigating the second charge, stating they have not confirmed its status. Customer requests that the second charge be removed if settled, but not canceled and seeks an update by August 13 to account for the August 14 payment due date."

### TODO - REFLECT 🖊

### Improvement and Human Review

**Effect of the revision:**

The revision **materially improved the outputs** by reducing unsupported claims and preserving more of the important case details. All three outputs correctly identified Elena Ruiz, order OR-8841, the August 6 and August 8 charges, the August 10 delivery, and the August 13/August 14 deadlines. However, some important details were still dropped, especially the **$186.40 amount in outputs 1 and 3** and the **case number CS-2197** in all three outputs. The model also made a slightly unsupported statement in output 3 that the order's delivery “demonstrates the charge status is accurate.”

**Role of the prompting strategy:**

The **few-shot strategy** helped demonstrate how to preserve financial details and distinguish confirmed information from unresolved information. This was particularly useful because the initial outputs incorrectly treated the second charge as a confirmed authorization. The example showed the model that uncertainty should be preserved rather than resolved or guessed.

**Human review still required:**

A billing employee must verify the **two $186.40 charges**, whether the second charge is actually settled or only a temporary authorization, and the status of any refund or removal. They should also verify the case number, payment deadlines, and account/payment records before taking action. The model should not independently decide whether a charge should be refunded, removed, or treated as fraudulent.


## Part 1.3: Drafting

**Business user:** Procurement analyst  
**Purpose:** Draft an internal delay notice that does not make unsupported commitments.

**Provided input**

```text
Supplier Northstar Filtration notified Procurement that shipment NF-771,
containing 12 replacement filters, will arrive August 19 instead of August 14.
The plant currently has approximately four days of filter inventory at normal
usage. Northstar offered expedited shipping for an additional fee, but
Procurement has not approved that option. Operations is checking whether usage
can be reduced safely. The plant manager needs a status update today. No
production shutdown has been scheduled.
```


**Your task:** A service-desk dispatcher needs to route this access request to the correct team and assign the correct urgency level. The routing and urgency definitions are included in the source text above — the model should apply them, not invent its own categories. The model should also flag that required information (employee ID and approval record) is missing.

**What to do in the cells below:**
- **First code cell:** Replace the `TODO` with your zero-shot instruction. Tell the model to output the routing category, urgency level, and any missing information that blocks processing, based strictly on the definitions provided. Run the cell and leave the output visible.
- **Diagnosis markdown cell:** Check whether the model applied the urgency definitions correctly (does the workaround change the urgency?), used only the defined categories, and flagged the missing fields. Identify the most significant gap.
- **Second code cell:** Write your revised instruction. Chain-of-thought is often effective here — prompting the model to reason through each definition before giving a final answer can reduce misclassification.
- **Evaluation markdown cell:** Explain what improved, what the strategy contributed, and what a human dispatcher must still decide before acting on the model's output.

### TODO - INSTRUCT 🔧

In [16]:
# 🔧 TODO - INSTRUCT
# Strategy: zero-shot — clear directions only, no examples.
initial_instruction_drafting = """
Strategy: zero-shot

Review the provided service request and determine how it should be routed.

Output exactly these three items:
- Routing Category:
- Urgency Level:
- Missing Information:

Use only the routing categories and urgency definitions explicitly provided in the source text. Do not invent, assume, or substitute your own categories or urgency levels.

Apply the urgency definition exactly as written, including whether a workaround or alternative solution changes the urgency level. Consider the current inventory, shipment delay, available workaround, and production impact only as described in the source.

Flag any required information that is missing and would prevent the request from being fully processed. In particular, check whether an employee ID and approval record are provided.

Keep the response concise and base every decision strictly on the provided information.
"""

In [17]:
# Run this to generate output after updating your instruction. It will take at least 20 seconds to run each loop.

for i in range(3):
  initial_record_drafting = run_part1(
      task_key="drafting",
      stage="initial",
      instruction=initial_instruction_drafting,
      source=PART1_SOURCES["drafting"],
  )
  display_record(initial_record_drafting)

**Model:** `gemma3:1b`  
**Recorded:** 2026-09-01T22:58:13+00:00  
**Elapsed:** 5.14 seconds

- Routing Category: Order Related
- Urgency Level: Medium - Immediate - Requires proactive action to understand the impact and coordinate with production. 
- Missing Information: Employee ID, Approval Record

**Model:** `gemma3:1b`  
**Recorded:** 2026-09-01T22:58:18+00:00  
**Elapsed:** 4.08 seconds

- Routing Category: Discrepancy Resolution - Filter Shipment
- Urgency Level: Medium
- Missing Information: Requires Employee ID & Approval Record for Operations.

**Model:** `gemma3:1b`  
**Recorded:** 2026-09-01T22:58:22+00:00  
**Elapsed:** 4.18 seconds

- Routing Cat: Lead Engnr. Resolution
- Urgency Level: Normal
-Missing Information: Employee ID, Approval Record, Production Impact details

### TODO - REFLECT 🖊

Initial-Output Diagnosis and Revision Plan

**Specific weakness in the initial output:**

The biggest weakness is that the model invented routing categories and urgency definitions instead of using only categories provided in the source. For example, the first output labeled the request “Order Related” and “Medium - Immediate,” while the second invented “Discrepancy Resolution - Filter Shipment,” and the third used “Lead Engnr. Resolution.” None of these categories appear in the provided source. The urgency levels also vary between “Medium - Immediate,” “Medium,” and “Normal,” even though the source says the shipment is delayed from August 14 to August 19, inventory is approximately four days, and Operations is checking whether usage can be safely reduced. The model did correctly identify Employee ID and Approval Record as missing.

**Planned instruction change:**

The revised instruction will require the model to reason through the urgency definitions and routing criteria step-by-step before producing the final classification. It will also explicitly prohibit inventing categories or treating missing definitions as if they were provided. The model should use the workaround information, Operations checking whether usage can be reduced safely, to determine whether the urgency changes.

**Prompting strategy for the revision:**

I will use chain-of-thought prompting because the main weakness is misclassification. Asking the model to work through each relevant definition and the effect of the workaround before giving its final answer should reduce inconsistent urgency decisions and help it apply the supplied criteria more carefully.


### Revise your prompt with a strategy to address the issue you noted above with the model output quality.

### TODO - INSTRUCT 🔧

In [18]:
# 🔧 TODO - INSTRUCT
# Identify your strategy at the start of the instruction string, e.g.: "Strategy: chain-of-thought"
revised_instruction_drafting = """
Strategy: chain-of-thought

Review the provided service request and determine the correct routing category and urgency level.

Before giving the final answer, reason through the routing and urgency criteria step by step. Consider the shipment delay, the approximately four days of remaining inventory, the unapproved expedited-shipping option, the possible reduction in usage, and the fact that no production shutdown has been scheduled.

Use only the routing categories and urgency definitions provided in the source. Do not invent, rename, or substitute categories or urgency levels. If the source does not actually provide enough information to determine the routing category or urgency level, state that clearly instead of guessing.

Also identify any required information that is missing. Specifically check whether an employee ID and approval record are provided.

Return exactly:
- Routing Category:
- Urgency Level:
- Missing Information:
- Brief Rationale:

Keep the final answer concise and use only information explicitly stated in the source.
"""

In [19]:
# Run this to generate output after updating your instruction.

revised_record_drafting = run_part1(
    task_key="drafting",
    stage="revised",
    instruction=revised_instruction_drafting,
    source=PART1_SOURCES["drafting"],
)
display_record(revised_record_drafting)

**Model:** `gemma3:1b`  
**Recorded:** 2026-09-01T22:59:58+00:00  
**Elapsed:** 4.74 seconds

Okay, let’s analyze this service request based purely on the provided text and source:

- **Routing Category:** Shipping Logistics
- **Urgency Level:** High
- **Missing Information:** Employee ID, Approval Record
- **Rationale:** The significant shipment delay (4 days of inventory) and the requirement for expedited shipping (even though Procurement hasn't approved) indicate a high-priority situation requiring proactive management of logistics and inventory.  The fact that Operations is checking usage reduction and the plant manager’s request for a status update support this assessment.

### TODO - REFLECT 🖊

Improvement and Human Review

**Effect of the revision:**
The revision somewhat improved the output because the model followed the requested format and provided a brief rationale. It also correctly identified the missing Employee ID and Approval Record. However, it still invented the “Shipping Logistics” category and “High” urgency level because the source did not actually provide routing or urgency definitions. This shows that the model can still make unsupported classifications when the required criteria are missing.

**Role of the prompting strategy:**
The chain-of-thought strategy encouraged the model to consider the shipment delay, inventory level, workaround, and production impact before making its decision. This helped produce a clearer rationale, but it did not prevent the model from inventing categories or urgency levels that were not provided in the source.

**Human review still required:**
A human must verify the routing category and urgency level against the organization’s actual definitions before taking action. The dispatcher should also confirm the missing Employee ID and Approval Record and determine whether the shipment delay could actually affect production.


# Part 2: Controlled Four-Model Evaluation

In Part 1, you were free to revise your instructions and iterate. Part 2 is different: you write **one** instruction and send it to all four models **unchanged**. This is a controlled experiment — the only variable is the model itself.

**Why controlled?** If you give different prompts to different models, any differences in output could come from your instruction, not from the model. A shared, unmodified prompt isolates the model as the only variable and makes your comparisons valid.



## Service Issue Details

```text
REQUEST SR-2401
Site: North Distribution Center
Reported by: Luis Ortega, extension 4410
At 6:40 a.m. on June 12, the quality-control freezer display read 18 F. Its
required operating range is 0-5 F. Temperature-sensitive calibration
materials were moved to the backup freezer. Staff reset the alarm twice,
but it returned both times. No employee injury was reported.
```


In [20]:
multimodeltext = """
REQUEST SR-2401
Site: North Distribution Center
Reported by: Luis Ortega, extension 4410
At 6:40 a.m. on June 12, the quality-control freezer display read 18 F. Its
required operating range is 0-5 F. Temperature-sensitive calibration
materials were moved to the backup freezer. Staff reset the alarm twice,
but it returned both times. No employee injury was reported.
"""

### TODO - INSTRUCT 🔧

In [21]:
# 🔧 TODO - INSTRUCT
# Strategy: zero-shot — clear directions only, no examples.
initial_instruction_extraction = """
Strategy: zero-shot

Extract the key information from the provided facilities service request and organize it into a consistent intake record.

Return the information using exactly these six fields:
- Reporter Name:
- Location:
- Problem Description:
- Date and Time Observed:
- Urgency or Deadline:
- Contact Information:

Use only information explicitly stated in the service request. Do not infer, assume, or invent information. Include all relevant details under the most appropriate field. Keep each field concise but specific.

For Urgency or Deadline, include only an explicit urgency, deadline, safety concern, or operational impact stated in the request. Do not assign an urgency level that is not provided.

Now extract the same six fields from the provided service request.
"""



In [22]:
# Run this after revising the

comparison_prompt = compose_prompt(
    initial_instruction_extraction,
    multimodeltext,
)

comparison_records = {}
for model in REQUIRED_MODELS:
    print(f"Running independent comparison: {model}")
    comparison_records[model] = chat_once(
        model,
        comparison_prompt,
        f"comparison_{model}",
    )
    display_record(comparison_records[model])


Running independent comparison: gemma3:1b


**Model:** `gemma3:1b`  
**Recorded:** 2026-09-01T23:02:33+00:00  
**Elapsed:** 4.69 seconds

Here's the intake record extracted from the provided service request, formatted according to your specification:

- Reporter Name: Luis Ortega
- Location: North Distribution Center
- Problem Description: freezer display read 18°F – requires 0-5°F range.
- Date and Time Observed: June 12, 2024, 6:40 a.m.
- Utergy or Deadline: None indicated
- Contact Information: Luis Ortega, extension 4410

Running independent comparison: gemma3:4b


**Model:** `gemma3:4b`  
**Recorded:** 2026-09-01T23:02:37+00:00  
**Elapsed:** 40.96 seconds

...

**Model:** `llama3.2:1b`  
**Recorded:** 2026-09-01T23:03:18+00:00  
**Elapsed:** 13.77 seconds

Here is the extracted information organized into a consistent intake record:

- Reporter Name: Luis Ortega
- Location: North Distribution Center
- Problem Description: Temperature-sensitive calibration materials were moved to the backup freezer and are out of range.
- Date and Time Observed: June 12, 6:40 a.m.
- Urgency or Deadline: Operational impact, safety concern ( temperature-sensitive materials)
- Contact Information: Luis Ortega, extension 4410

Running independent comparison: llama3.2:3b


**Model:** `llama3.2:3b`  
**Recorded:** 2026-09-01T23:03:32+00:00  
**Elapsed:** 26.43 seconds

Here is the extracted information organized into a consistent intake record using the specified six fields:

- Reporter Name: Luis Ortega
- Location: North Distribution Center
- Problem Description: The quality-control freezer display read 18 F, outside its required operating range of 0-5 F.
- Date and Time Observed: June 12, 6:40 a.m.
- Urgency or Deadline: Temperature-sensitive calibration materials were moved to the backup freezer.
- Contact Information: Extension 4410 (Luis Ortega)

### TODO - REFLECT 🖊

You can fill these in like this:

### Model Evaluation

**How similar or different were the outputs?**
The outputs were generally similar because all four models correctly identified Luis Ortega, the North Distribution Center, the freezer temperature, and the June 12 observation time. However, they differed in how much detail they included and whether they inferred urgency. For example, gemma3:4b included that the alarm returned after being reset, while llama3.2:3b included that the temperature-sensitive materials were moved to the backup freezer. Llama3.2:1b also labeled the situation as an “operational impact” and “safety concern,” even though those terms were not explicitly stated in the request. Gemma3:1b incorrectly added the year “2024,” which was not provided.

**How long did each model take to run?**
The models varied significantly in speed. Gemma3:1b was fastest at **4.69 seconds**, followed by llama3.2:1b at **13.77 seconds**, llama3.2:3b at **26.43 seconds**, and gemma3:4b was slowest at **40.96 seconds**. Overall, the larger 3b/4b models took longer to run than the smaller 1b models.

**Which model was "Best"?**
I would choose **gemma3:4b** as the best model for this task. Although it took the longest, its output was accurate, followed the six-field format, included important details such as the alarm returning after reset, and did not invent an urgency level or date. The differences are substantial enough to justify choosing it when accuracy and completeness are more important than processing speed.


# Part 3: Reflect on the Assignment



#### TODO - FINAL REFLECTION 🖊

The revision that produced the largest improvement was the Part 1.2 summarization task. In the initial version, the model made serious mistakes with financial information. One output incorrectly calculated the total charges, while another described the second charge as a pending authorization even though the source specifically said Billing had not confirmed whether it was a settled charge or a temporary authorization. The revised prompt told the model to preserve every amount, date, case number, and deadline exactly and to clearly separate confirmed information from unresolved information. This reduced the number of unsupported claims and made the summaries more appropriate for a billing team. However, the model still sometimes left out important details, showing why human review is necessary.

For strategy fit, I used few-shot prompting when the task benefited from showing the model exactly what the desired output should look like. This was especially useful for extraction and summarization because the examples demonstrated both the format and the level of detail expected. I used chain-of-thought for the classification task because the model needed to consider several factors, such as the shipment delay, remaining inventory, available workaround, and production impact before determining urgency. Zero-shot prompting was sufficient for the freezer extraction because the six required fields and instructions were already clear, so an example was not necessary.

In Part 2, the models differed most noticeably in completeness and their tendency to make unsupported assumptions. For example, llama3.2:1b called the freezer situation an “operational impact” and “safety concern,” even though those terms were not explicitly stated. Gemma3:1b also incorrectly added the year 2024 to the date. In comparison, gemma3:4b stayed closer to the source and included the important detail that the alarm returned after being reset.

The larger model in each family did not always produce better results simply because it was larger. However, gemma3:4b produced the strongest overall output for the freezer task, while llama3.2:3b also gave a detailed and accurate response. This suggests that model size can help, but it does not guarantee better performance.

Runtime was another important difference. Gemma3:1b took only 4.69 seconds, while gemma3:4b took 40.96 seconds. Llama3.2:1b took 13.77 seconds and llama3.2:3b took 26.43 seconds. For simple, repetitive extraction, the smaller models may be worth the speed. However, for financial or operational decisions, the additional time for a larger model may be justified by better accuracy and completeness.

One major business risk would be acting on the incorrect financial information from the original Part 1.2 output. A false total or unsupported claim about a charge could lead Billing to issue an incorrect refund or make an incorrect account decision. A human should verify all financial amounts, transaction statuses, and case information before taking action.

Overall, small local models are reasonable for low-risk, repetitive tasks with clear instructions and structured information. For tasks involving money, safety, compliance, or important business decisions, I would prefer a larger or cloud-hosted model, combined with human review.

